<div style="background:#102A43;color:#ffffff;border-radius:11px;padding:30px 28px 24px;border-top:7px solid #18B7AC;">
<div style="color:#67DED3;letter-spacing:2px;font-weight:700;font-size:12px;">INTRODUCTION TO MACHINE LEARNING · GAZA SKY GEEKS</div>
<div style="font-size:32px;font-weight:750;line-height:1.25;margin:12px 0 9px;">Lecture 09 · Measuring Similarity:<br>Distances and Proximity Measures</div>
<div style="font-size:15px;color:#DDEAF3;">Instructor: Diaeddin Rimawi &nbsp; | &nbsp; 14 November 2026 &nbsp; | &nbsp; Part 2: Data Preprocessing and Proximity Measures</div>
<div style="font-size:17px;color:#ffffff;border-top:1px solid #466178;padding-top:16px;margin-top:18px;"><strong>Today's challenge:</strong> We have numerical representations of our penguins. Which penguin is most similar to a selected penguin—and could a different way of measuring similarity change our answer?</div>
</div>

We will explore **proximity**, not train a predictive model.

### What you will be able to do

By the end of this lecture, you should be able to:

- Distinguish **distance/dissimilarity** from **similarity**, and describe what *proximity* means.
- Interpret an observation as a **feature vector** and calculate **Euclidean** and **Manhattan** distances.
- Explain why different measures can produce different **nearest-observation rankings**.
- Find nearby observations using **NumPy** and **Pandas**, without training a model.
- Demonstrate how **units and standardization** can change numerical proximity.
- Explain how **cosine similarity** compares *direction* rather than *magnitude*.
- Recognize why **categorical attributes** and **redundant features** need careful treatment.
- Explain why a close neighbor does **not** automatically imply the same species.

**Guiding idea:** A distance calculation may be correct, but whether it is *meaningful* depends on our features, representation, and chosen measure.

### From L08 to L09: what can we do with our numerical features?

In **L08**, we learned how to transform penguin observations into a numerical feature table:

- Measurements can be in different **units**; scaling changes their numerical representation.
- A **bill-shape ratio** can describe information already related to bill length and depth.
- **Island** can be encoded as indicator columns instead of arbitrary integer codes.
- The intended target, **`species`**, must remain separate from the input features.
- Parameters learned from data belong to the **training** subset in a future prediction workflow.

**Opening question:** If two penguins have different measurements, how can we combine those differences into one meaningful comparison?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">01 · WHAT DOES SIMILARITY MEAN?</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">We must decide what counts as close before we calculate it.</div>
</div>

### Similar according to what?

Imagine three penguins:

- Two may have nearly identical **bill measurements**.
- Another may have a more similar **body mass**.
- Two may live on the **same island**.

Which pair is more similar? There is no single answer until we explain *what we are comparing*.

| Term | General interpretation | Typical direction |
|:--|:--|:--|
| **Distance** | How far apart two feature representations are | Smaller = closer |
| **Dissimilarity** | How different two observations are | Smaller = more alike |
| **Similarity** | How strongly two representations resemble each other | Larger = more alike |
| **Proximity** | Broad term for closeness or resemblance | Depends on its measure |

These words are related, but a similarity score is **not automatically** `1 − distance`. Different measures have different definitions and numerical ranges.

### Prepare our familiar dataset (without changing it)

We use the same **344-row Palmer Penguins CSV** as L07 and L08. Two observations are missing one or more physical measurements, so our complete-measurement working table contains **342 rows**.

- For now we do **not** need `sex`, `year`, `island`, or `species` to calculate numerical distance.
- We preserve the original **row indices** so that we can identify real observations unambiguously.
- Species labels may be inspected **after** computing distances, but will never be used as proximity inputs.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

NAVY = "#102A43"
TEAL = "#18B7AC"
SPECIES_ORDER = ["Adelie", "Chinstrap", "Gentoo"]
SPECIES_COLORS = {"Adelie": "#158B94", "Chinstrap": "#DB9137", "Gentoo": "#557AB5"}
MEASUREMENTS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "figure.dpi": 110})

data_path = Path("data/penguins.csv")
if not data_path.is_file():
    raise FileNotFoundError(
        "Cannot find data/penguins.csv. Put the existing course CSV inside "
        "a data/ folder next to this notebook (as in L08)."
    )
penguins = pd.read_csv(data_path)
assert penguins.shape == (344, 8), "Unexpected penguins.csv structure"
study = penguins.dropna(subset=MEASUREMENTS).copy()
assert len(study) == 342

print("Original dataset:", penguins.shape)
print("Rows with all four physical measurements:", study.shape[0])
display(study.loc[[0, 145, 293], ["bill_length_mm", "bill_depth_mm",
                                    "flipper_length_mm", "body_mass_g", "island"]])

### Each observation becomes a point in a feature space

If we select **bill length** and **bill depth**, a penguin can be represented by two coordinates:

$$
\mathbf{x}=(\text{bill length},\ \text{bill depth}).
$$

- One measurement corresponds to one **dimension** of the feature space.
- Two features make a point in a **two-dimensional plane**.
- Four numerical features make a point in a **four-dimensional** space. We can calculate distances there even though we cannot draw all dimensions at once.

Let's first look at three observations (their original row indices are shown). Based on bill measurements alone, which appears closest to row 0?

In [ ]:
fig, ax = plt.subplots(figsize=(9.1, 4.8))
ax.scatter(study["bill_length_mm"], study["bill_depth_mm"],
           color="#C7D4DB", s=25, alpha=0.6, label="Other observations")
for idx, color in [(0, NAVY), (145, TEAL), (293, "#DB9137")]:
    row = study.loc[idx]
    ax.scatter(row["bill_length_mm"], row["bill_depth_mm"], s=135,
               color=color, edgecolor="white", linewidth=1.4, zorder=5)
    ax.annotate(f"row {idx}", (row["bill_length_mm"], row["bill_depth_mm"]),
                xytext=(7, 8 if idx != 145 else -18), textcoords="offset points",
                color=color, fontsize=10, weight="bold")
ax.set(xlabel="Bill length (mm)", ylabel="Bill depth (mm)",
       title="Three penguins represented by two physical measurements")
ax.grid(alpha=0.17)
plt.show()

**Notice:** Row 145 almost overlaps row 0 in this view, while row 293 is much farther to the right. That is only a statement about **these two bill measurements**.

**Next question:** Can we turn the apparent gap between two points into a numerical distance?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">02 · EUCLIDEAN DISTANCE</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">From a right triangle to numerical feature vectors.</div>
</div>

### Begin with a familiar geometric problem

Take two points (these are not penguins):

$$
A=(1,2), \qquad B=(4,6).
$$

1. Moving horizontally from $A$ to $B$ changes the first coordinate by $4-1=3$.
2. Moving vertically changes the second coordinate by $6-2=4$.
3. These changes are the sides of a right triangle.
4. By the **Pythagorean theorem**, the straight-line distance is:

$$
\boxed{d_E(A,B)=\sqrt{3^2+4^2}=\sqrt{25}=5.}
$$

This is **Euclidean distance**: the ordinary straight-line separation in a numerical coordinate space.

In [ ]:
A = np.array([1.0, 2.0])
B = np.array([4.0, 6.0])
coordinate_differences = B - A
squared_differences = coordinate_differences ** 2
euclidean_distance = np.sqrt(np.sum(squared_differences))

print("Coordinate differences (B - A):", coordinate_differences)
print("Squared differences:", squared_differences)
print("Sum of squares:", squared_differences.sum())
print("Euclidean distance:", euclidean_distance)
print("NumPy's vector length gives the same result:", np.linalg.norm(B - A))

fig, ax = plt.subplots(figsize=(6.5, 5.2))
ax.plot([A[0], B[0]], [A[1], A[1]], linestyle="--", color=TEAL, lw=2)
ax.plot([B[0], B[0]], [A[1], B[1]], linestyle="--", color=TEAL, lw=2)
ax.plot([A[0], B[0]], [A[1], B[1]], color="#557AB5", lw=3,
        label="Straight-line (Euclidean) distance")
ax.scatter([A[0], B[0]], [A[1], B[1]], s=110, color=[NAVY, TEAL], zorder=4)
ax.annotate("A (1, 2)", A, xytext=(-18, -22), textcoords="offset points", weight="bold")
ax.annotate("B (4, 6)", B, xytext=(7, 7), textcoords="offset points", weight="bold")
ax.text(2.3, 1.75, "3 units", color=TEAL)
ax.text(4.1, 3.6, "4 units", color=TEAL)
ax.text(2.0, 4.4, "distance = 5", color="#557AB5", weight="bold")
ax.set(xlim=(0, 6), ylim=(0, 7), xlabel="Feature 1", ylabel="Feature 2",
       title="Euclidean distance comes from Pythagoras")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.16)
plt.show()

### Generalize the same idea to several features

Suppose two observations have $n$ **numerical** features:

$$
A=(a_1,a_2,\ldots,a_n),\quad B=(b_1,b_2,\ldots,b_n).
$$

The straight-line distance is:

$$
\boxed{d_E(A,B)=\sqrt{\sum_{i=1}^{n}(a_i-b_i)^2}.}
$$

**Read the expression step by step:**

- $a_i-b_i$: difference in the **same feature** for two observations.
- Square each difference so positive and negative differences cannot cancel.
- Add the squared differences across all $n$ features.
- Take the square root to return to the Euclidean distance.

This formula works for **two, four, or many dimensions**. Each compared feature must have a meaningful numerical representation; simply having a numeric data type is not enough.

### Now compare two real penguins

Using **bill length and bill depth**, row 0 has measurements (39.1, 18.7) mm, and row 145 has (39.0, 18.7) mm.

- Their bill depths match exactly.
- Their bill lengths differ by just **0.1 mm**.
- In this two-feature representation, their Euclidean distance is therefore **0.1 mm**.

We did not use species labels to make the calculation.

In [ ]:
BILL_FEATURES = ["bill_length_mm", "bill_depth_mm"]
penguin_A = study.loc[0, BILL_FEATURES].to_numpy(dtype=float)
penguin_B = study.loc[145, BILL_FEATURES].to_numpy(dtype=float)
print("Row 0:", penguin_A)
print("Row 145:", penguin_B)
print("Differences:", penguin_A - penguin_B)
print(f"Euclidean distance (bill measurements): {np.linalg.norm(penguin_A - penguin_B):.3f} mm")

**Important interpretation:** We have calculated distance **between feature vectors**, not physical distance between penguins on an island. Two penguins whose bills are alike may be geographically far apart, and they may still differ in other attributes.

**Next question:** What happens if we choose another mathematically reasonable definition of distance?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">03 · MANHATTAN DISTANCE</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Different ways of measuring a gap can produce different neighbors.</div>
</div>

### What if we move only along the coordinate axes?

Imagine moving through a city where streets form a rectangular grid. From $A=(1,2)$ to $B=(4,6)$, we could walk:

- 3 units in one direction, and
- 4 units in the other direction.

The total is $3+4=7$, even though the straight-line distance is 5.

**Manhattan distance** adds the **absolute differences**:

$$
\boxed{d_M(A,B)=\sum_{i=1}^{n}|a_i-b_i|.}
$$

Absolute value makes a difference nonnegative: $|-3|=3$. There is **no squaring and no square root**.

In [ ]:
manhattan_distance = np.abs(B - A).sum()
print("Euclidean distance A to B:", np.linalg.norm(B - A))
print("Manhattan distance A to B:", manhattan_distance)

# New controlled comparison: the 'nearest' point can change.
origin = np.array([0.0, 0.0])
point_P = np.array([3.0, 0.0])
point_Q = np.array([2.0, 2.0])

rows = []
for name, point in [("P = (3, 0)", point_P), ("Q = (2, 2)", point_Q)]:
    difference = point - origin
    rows.append({"Candidate": name,
                 "Euclidean": np.sqrt(np.sum(difference ** 2)),
                 "Manhattan": np.abs(difference).sum()})
controlled_comparison = pd.DataFrame(rows).set_index("Candidate")
display(controlled_comparison.round(3))
print("Closest under Euclidean:", controlled_comparison["Euclidean"].idxmin())
print("Closest under Manhattan:", controlled_comparison["Manhattan"].idxmin())

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), constrained_layout=True)
for ax in axes:
    ax.scatter([0, 3, 2], [0, 0, 2], s=115, color=[NAVY, "#DB9137", TEAL], zorder=5)
    for pt, label, offset in [(origin, "A", (-12, -19)), (point_P, "P", (7, -14)),
                              (point_Q, "Q", (7, 8))]:
        ax.annotate(label, pt, xytext=offset, textcoords="offset points", weight="bold")
    ax.set(xlim=(-0.5, 3.8), ylim=(-0.5, 3.0), xlabel="Feature 1", ylabel="Feature 2")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks(np.arange(0, 4)); ax.set_yticks(np.arange(0, 4)); ax.grid(alpha=0.18)
axes[0].plot([0, 3], [0, 0], color="#DB9137", lw=2.7)
axes[0].plot([0, 2], [0, 2], color=TEAL, lw=2.7)
axes[0].set_title("Euclidean: Q is closer (2.83 < 3)")
axes[1].plot([0, 3], [0, 0], color="#DB9137", lw=2.7)
axes[1].plot([0, 2, 2], [0, 0, 2], color=TEAL, lw=2.7)
axes[1].set_title("Manhattan: P is closer (3 < 4)")
plt.show()

### Same points, different ranking

Notice what happened in our **controlled example**:

| Measure | Distance to P | Distance to Q | Closer candidate |
|:--|--:|--:|:--|
| Euclidean | 3 | $\sqrt{8}\approx 2.83$ | Q |
| Manhattan | 3 | 4 | P |

Neither calculation is mistaken. **The definition of distance determines the ranking.**

A few mathematical facts characterize a true distance **metric**:

- A distance cannot be negative.
- The distance is zero *exactly when the represented points are identical*.
- Going from A to B is the same distance as going from B to A (**symmetry**).
- A direct route is no longer than a route via an intermediate point (**triangle inequality**).

Euclidean and Manhattan satisfy these properties. We do not need formal proofs today.

### A short connection: Minkowski distance

Euclidean and Manhattan are two members of a larger family:

$$
d_p(A,B)=\left(\sum_{i=1}^{n}|a_i-b_i|^p\right)^{1/p},\qquad p\ge 1.
$$

- With $p=1$, the formula becomes **Manhattan distance**.
- With $p=2$, it becomes **Euclidean distance**.

You do not need to memorize or experiment with other values of $p$ in this lecture. Our goal is to understand what the first two measures *mean*.

**Next question:** Can we use these formulas to find a real penguin's nearest observations?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">04 · A CONTINUOUS PROXIMITY INVESTIGATION</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Find the closest penguins.</div>
</div>

### Choose a reference observation and compare it with all others

We return to the **342 complete-measurement records** and use two bill measurements for clarity.

Our plan is deliberately simple:

1. Choose a **reference penguin**: original row `0`.
2. Calculate its Euclidean distance to **every row** using bill length and depth.
3. Remove the reference row itself; its distance to itself is zero.
4. Sort the distances and display the **five closest observations**.
5. Inspect the results and then compare Manhattan rankings.

This is a **descriptive proximity search**, not classification. We will not train a model or use neighbor labels to predict species.

In [ ]:
reference_index = 0
reference_vector = study.loc[reference_index, BILL_FEATURES].to_numpy(dtype=float)
bill_matrix = study[BILL_FEATURES].to_numpy(dtype=float)

# Broadcasting: subtract the reference vector from every row.
bill_differences = bill_matrix - reference_vector
bill_euclidean = np.sqrt(np.sum(bill_differences ** 2, axis=1))
bill_distances = pd.Series(bill_euclidean, index=study.index, name="euclidean_mm")

# Exclude the reference itself: a zero distance to itself is not a new neighbor.
nearest_bill = bill_distances.drop(index=reference_index).sort_values(kind="stable").head(5)
nearest_bill_table = study.loc[nearest_bill.index, BILL_FEATURES].copy()
nearest_bill_table.insert(0, "distance_mm", nearest_bill)
print("Reference row:", reference_index)
display(study.loc[[reference_index], BILL_FEATURES])
print("Five nearest observations using bill length and bill depth:")
display(nearest_bill_table.round(3))

**Interpretation:** The closest row has the smallest computed distance *in these two bill measurements*. It is not necessarily the closest using mass, flipper length, island, or all features together.

We can **color the resulting points by their known species for interpretation**—but that label was **not** an input to the proximity calculation. That distinction matters.

In [ ]:
# Left: global view. Right: close-up so individual near-neighbors are readable.
fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.9), constrained_layout=True)
for ax in axes:
    for species in SPECIES_ORDER:
        subset = study.loc[study["species"] == species]
        ax.scatter(subset["bill_length_mm"], subset["bill_depth_mm"],
                   color=SPECIES_COLORS[species], alpha=0.35, s=28, label=species)
    ax.grid(alpha=0.15)
    ax.set(xlabel="Bill length (mm)", ylabel="Bill depth (mm)")

ref = study.loc[reference_index]
axes[0].scatter([ref["bill_length_mm"]], [ref["bill_depth_mm"]],
                marker="*", color=NAVY, edgecolor="white", s=240,
                linewidth=0.9, zorder=7, label="Reference (row 0)")
axes[0].set_title("Overview: three penguin species")
axes[0].legend(frameon=False, fontsize=9, loc="lower right")

# Label only the five nearest on the zoomed view, where their differences are visible.
zoom = axes[1]
label_offsets = {145: (-27, -19), 146: (8, -15), 89: (-28, 11),
                 16: (-26, 6), 45: (7, 7)}
for idx in nearest_bill.index:
    neighbor = study.loc[idx]
    zoom.plot([ref["bill_length_mm"], neighbor["bill_length_mm"]],
              [ref["bill_depth_mm"], neighbor["bill_depth_mm"]],
              color=NAVY, lw=1.3, alpha=0.72, zorder=3)
    zoom.scatter([neighbor["bill_length_mm"]], [neighbor["bill_depth_mm"]],
                 s=90, color=TEAL, edgecolor="white", linewidth=0.8, zorder=6)
    zoom.annotate(f"{idx}", (neighbor["bill_length_mm"], neighbor["bill_depth_mm"]),
                  xytext=label_offsets.get(idx, (7, 7)), textcoords="offset points",
                  fontsize=10, weight="bold", color=NAVY)
zoom.scatter([ref["bill_length_mm"]], [ref["bill_depth_mm"]],
             marker="*", s=250, color=NAVY, edgecolor="white", linewidth=0.8, zorder=8)
zoom.annotate("reference 0", (ref["bill_length_mm"], ref["bill_depth_mm"]),
              xytext=(5, -22), textcoords="offset points", fontsize=9, weight="bold")
zoom.set(xlim=(38.35, 40.0), ylim=(18.25, 19.25),
         title="Close-up: five nearest observations")
plt.show()

### Are close observations guaranteed to have the same species?

No. Physical measurements from different species can overlap. A small distance tells us that **selected measurements are alike**, not that the biological labels must match.

We can inspect species **after** we have found neighbors. This does not make the species label a distance feature.

In [ ]:
interpreted_neighbors = pd.DataFrame({
    "distance_mm": nearest_bill,
    "species_for_interpretation": study.loc[nearest_bill.index, "species"],
    "island_for_interpretation": study.loc[nearest_bill.index, "island"],
})
display(interpreted_neighbors.round(3))

# Same reference, same two features, but a different measure.
bill_manhattan = pd.Series(np.abs(bill_differences).sum(axis=1),
                           index=study.index, name="manhattan_mm")
nearest_manhattan = bill_manhattan.drop(index=reference_index).sort_values(kind="stable").head(5)
comparison = pd.DataFrame({
    "Euclidean: nearest row": nearest_bill.index.tolist(),
    "Manhattan: nearest row": nearest_manhattan.index.tolist(),
}, index=pd.Index(range(1, 6), name="Rank"))
display(comparison)
print("Same five rows in both lists?", set(nearest_bill.index) == set(nearest_manhattan.index))

**What changed?** Even for the same reference and the same two features, Manhattan and Euclidean can change the **order of nearby observations**. In other examples (as our three-point demonstration showed), they can even choose a different nearest observation.

**Next question:** All our bill measurements were in **millimeters**. What happens when we mix features recorded on very different numerical scales?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">05 · WHEN UNITS CHANGE THE ANSWER</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">A different coordinate representation can change the nearest neighbor.</div>
</div>

### Compare flipper length and body mass

This time we use **flipper length (mm)** and **body mass (g)**. These measurements have very different numerical sizes.

For three real observations in the CSV:

| Original row | Flipper length (mm) | Body mass (g) |
|:--|--:|--:|
| **0 — reference** | 181 | 3,750 |
| **149 — candidate** | 193 | 3,750 |
| **293 — candidate** | 181 | 3,700 |

Look at the differences from row 0:

- Row 149 differs by **12 mm** in flipper length, but has the **same mass**.
- Row 293 has the **same flipper length**, but differs by **50 g** in mass.

**Prediction:** Which candidate will have the smaller numerical Euclidean distance using raw mm and g? And what might happen if mass is recorded in **kilograms** instead?

In [ ]:
MASS_FLIPPER_FEATURES = ["flipper_length_mm", "body_mass_g"]
three_penguins = study.loc[[0, 149, 293], MASS_FLIPPER_FEATURES].copy()
display(three_penguins)

# Reminder: distances combine *numerical coordinates*; when units differ,
# their sizes cannot be interpreted as a single physical length or mass.
print("Reference row:", three_penguins.index[0])
print("Candidate rows:", three_penguins.index[1:].tolist())

<div style="border-left:5px solid #18B7AC;background:#EEF8F7;padding:12px 16px;border-radius:5px;"><strong style="color:#102A43;">ACTIVITY 1 — Did the closest penguin move?</strong></div>

**Your challenge:** Use the three real observations just introduced, and apply the Euclidean calculation we already practiced—but now **change the unit of one feature**.

1. Compute the Euclidean distances from **row 0** to rows **149** and **293** using `flipper_length_mm` and `body_mass_g`.
2. Make a **copy** of the three-row table, convert its mass column from grams to **kilograms**, and repeat the distances.
3. Identify the closer candidate in each representation.
4. Explain: **Did the penguins change, or did our definition of numerical proximity change?**

**Hint:** Subtract each candidate vector from the reference vector, square the differences, add them, and take the square root. Use `.copy()` so you preserve the original observations.

**Discovery to look for:** A harmless change in recorded units can change which observation appears closest under a raw distance formula.

In [ ]:
# Activity 1 — The table is ready; add your calculations below.
activity1 = three_penguins.copy()
reference_row = 0
candidate_rows = [149, 293]

# TODO 1: Calculate Euclidean distances from row 0 in raw mm and g.
# TODO 2: Copy activity1, convert body_mass_g / 1000, and recalculate.
# TODO 3: Compare the closest candidate before and after the conversion.

# This starter cell runs unchanged; use the space above to experiment.
display(activity1)
print("Work with reference", reference_row, "and candidates", candidate_rows)

<details>
<summary><strong>Suggested answer — click to expand after trying</strong></summary>

**Suggested code**

```python
activity1 = three_penguins.copy()
reference_row = 0
candidate_rows = [149, 293]

# Compute the same Euclidean formula for each candidate.
def activity_distances(frame):
    differences = frame.loc[candidate_rows] - frame.loc[reference_row]
    return np.sqrt((differences ** 2).sum(axis=1))

raw_distances = activity_distances(activity1)
activity1_kg = activity1.copy()
activity1_kg["body_mass_g"] = activity1_kg["body_mass_g"] / 1000
kg_distances = activity_distances(activity1_kg)

result = pd.DataFrame({"raw_mm_and_g": raw_distances,
                       "mm_and_kg": kg_distances})
display(result.round(3))
print("Closer with grams:", raw_distances.idxmin())
print("Closer with kilograms:", kg_distances.idxmin())
```

**Expected discovery:**

- **Raw millimeters and grams:** row 149 has distance **12**, while row 293 has distance **50**. Row **149** appears closer.
- **Millimeters and kilograms:** row 149 still has distance **12**, while row 293 has distance **0.05**. Row **293** appears closer.
- Only the **numbers used as coordinates** changed. We did not change the observations, and the result is not a change in biological similarity.

**Interpretation:** The raw Euclidean formula effectively gives more influence to features with larger numerical differences. The combined values here have **mixed measurement units**, so do not describe the resulting scalar as an ordinary physical distance in mm, g, or kg.

**Bridge:** If changing units can reverse the comparison for two candidates, could it also change the closest observation among **all** 342 penguins?

</details>

### From two candidates to all observations

Let's now apply the same idea to the full working table, **using row 0 as before**.

We calculate the nearest observations once with mass in grams, then again with mass in kilograms. We keep all observations and their original indices unchanged.

In [ ]:
mass_flipper_g = study[MASS_FLIPPER_FEATURES].copy()
mass_flipper_kg = mass_flipper_g.copy()
mass_flipper_kg["body_mass_g"] = mass_flipper_kg["body_mass_g"] / 1000

def euclidean_to_reference(frame, ref_index=0):
    values = frame.to_numpy(dtype=float)
    reference = frame.loc[ref_index].to_numpy(dtype=float)
    distance = np.sqrt(np.sum((values - reference) ** 2, axis=1))
    return pd.Series(distance, index=frame.index, name="distance")

all_distances_g = euclidean_to_reference(mass_flipper_g)
all_distances_kg = euclidean_to_reference(mass_flipper_kg)
nearest_g = all_distances_g.drop(index=0).sort_values(kind="stable").head(5)
nearest_kg = all_distances_kg.drop(index=0).sort_values(kind="stable").head(5)

display(pd.DataFrame({
    "Nearest rows using mass in grams": nearest_g.index.tolist(),
    "Nearest rows using mass in kilograms": nearest_kg.index.tolist(),
}, index=pd.Index(range(1, 6), name="Rank")))
print("Closest with mass in grams:", nearest_g.index[0])
print("Closest with mass in kilograms:", nearest_kg.index[0])

In [ ]:
# A compact comparison of the two candidate distances in the activity.
candidate_ids = [149, 293]
fig, axes = plt.subplots(1, 2, figsize=(10.6, 3.7), constrained_layout=True)
for ax, ds, label in [(axes[0], all_distances_g, "Flipper mm + mass g"),
                      (axes[1], all_distances_kg, "Flipper mm + mass kg")]:
    values = ds.loc[candidate_ids].to_numpy()
    bars = ax.bar(["Row 149", "Row 293"], values, color=["#557AB5", TEAL], width=0.58)
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=10)
    ax.set(title=label, ylabel="Numerical Euclidean distance", ylim=(0, max(values) * 1.2))
    ax.grid(axis="y", alpha=0.16)
plt.show()

### Standardization can reduce arbitrary unit effects—but it is a choice

Remember the **z-score** from L08:

$$
z=\frac{x-\mu_{\text{train}}}{\sigma_{\text{train}}}.
$$

For a future prediction workflow, learn the reference **mean and sample standard deviation from training rows only**, then transform validation, test, and future observations using those saved statistics.

We will use an **illustrative split** (fixed random seed) only to obtain reproducible training statistics for this lecture. It is **not** a replacement for the split assignments 03 and 04.

We first standardize the same **two features**, and also save a **four-measurement** representation to investigate later.

In [ ]:
# Illustrative split used only to obtain leakage-safe reference statistics.
rng = np.random.default_rng(8)
shuffled_indices = rng.permutation(study.index.to_numpy())
n_train = int(0.70 * len(study))
n_validation = int(0.20 * len(study))
split_role = pd.Series("test", index=study.index, name="split_role")
split_role.loc[shuffled_indices[:n_train]] = "train"
split_role.loc[shuffled_indices[n_train:n_train + n_validation]] = "validation"
training_rows = split_role == "train"

training_means = study.loc[training_rows, MEASUREMENTS].mean()
training_sds = study.loc[training_rows, MEASUREMENTS].std(ddof=1)
assert (training_sds > 0).all()

numeric_z = (study[MEASUREMENTS] - training_means) / training_sds
z_two = numeric_z[MASS_FLIPPER_FEATURES]
nearest_z_two = (euclidean_to_reference(z_two)
                 .drop(index=0).sort_values(kind="stable").head(5))

print("Illustrative train/validation/test counts:", split_role.value_counts().to_dict())
print("Closest with two standardized measurements:", nearest_z_two.index[0])
display(pd.DataFrame({"distance_in_z_score_space": nearest_z_two}).round(3))

### A useful test: what if we standardize grams and kilograms separately?

If every mass is consistently converted from grams to kilograms, both its **training mean** and **training standard deviation** are divided by 1,000.

For each penguin, the resulting z-score is therefore **unchanged**. This is one advantage of standardizing in a consistent reference system.

In [ ]:
mass_kg = study["body_mass_g"] / 1000
train_mean_kg = mass_kg.loc[training_rows].mean()
train_sd_kg = mass_kg.loc[training_rows].std(ddof=1)
z_mass_kg = (mass_kg - train_mean_kg) / train_sd_kg

print("Do mass z-scores match after converting g to kg?",
      np.allclose(numeric_z["body_mass_g"], z_mass_kg))
print("Largest numerical difference:",
      np.max(np.abs(numeric_z["body_mass_g"] - z_mass_kg)))

# Important: standardizing does not decide which features are relevant,
# remove duplicates, or make every distance measure equally appropriate.

**What we have discovered:**

- Raw distance can be **dominated by units or numerical scale**.
- Standardization changes the comparison to differences measured in **training standard deviations**.
- Standardization does **not guarantee** useful similarity. Irrelevant or repeated features can still influence results.
- Not every machine learning algorithm needs feature scaling; its importance depends on the algorithm and representation.

**Next question:** We have been comparing sizes and differences. Could two observations be similar because their *patterns point in the same direction*, even if one is much larger?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">06 · COSINE SIMILARITY</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Sometimes the direction of a vector matters more than its length.</div>
</div>

### Same pattern, very different magnitude

Imagine two simple **word-frequency vectors**. Each coordinate counts a word in a tiny document:

- **Word 1:** “data”
- **Word 2:** “learning”

Consider three documents:

| Document | `data` count | `learning` count |
|:--|--:|--:|
| **A** | 1 | 2 |
| **B** | 10 | 20 |
| **C** | 2 | 1 |

Document B is much longer than A, but has exactly the same **relative word pattern**. Document C has different proportions.

**Question:** If relative pattern matters more than total length, should A be considered more similar to B or C?

### Cosine compares angles rather than absolute magnitudes

Think of a vector as an **arrow from the origin**.

- Two arrows pointing in the **same direction** have cosine similarity **1**.
- Two arrows at a **right angle** have cosine similarity **0**.
- Arrows pointing in **opposite directions** have cosine similarity **−1**.

For two **nonzero** vectors:

$$
\boxed{\operatorname{cos\_sim}(A,B)=\frac{A\cdot B}{\|A\|\,\|B\|}.}
$$

Here, $A\cdot B$ is the **dot product** (multiply matching coordinates and add), while $\|A\|$ and $\|B\|$ are the **lengths** of the two vectors.

For word counts, all coordinates are nonnegative, so cosine similarity is between **0 and 1**. The zero vector has no direction, so its cosine similarity is **undefined** without an explicit convention.

In [ ]:
documents = pd.DataFrame({"data": [1, 10, 2], "learning": [2, 20, 1]},
                         index=["A", "B", "C"], dtype=float)
display(documents.astype(int))

def cosine_similarity(a, b):
    length_product = np.linalg.norm(a) * np.linalg.norm(b)
    if length_product == 0:
        raise ValueError("Cosine similarity is undefined for a zero vector")
    return np.dot(a, b) / length_product

word_A = documents.loc["A"].to_numpy()
word_B = documents.loc["B"].to_numpy()
word_C = documents.loc["C"].to_numpy()

word_comparison = pd.DataFrame({
    "Euclidean distance from A": [np.linalg.norm(word_A - word_B),
                                  np.linalg.norm(word_A - word_C)],
    "Cosine similarity to A": [cosine_similarity(word_A, word_B),
                               cosine_similarity(word_A, word_C)],
}, index=["B", "C"])
display(word_comparison.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(7.7, 5.0))
for name, end, color in [("A", word_A, NAVY),
                         ("B", word_B, TEAL),
                         ("C", word_C, "#DB9137")]:
    ax.annotate("", xy=end, xytext=(0, 0),
                arrowprops={"arrowstyle": "->", "color": color, "lw": 2.7})
    offset = (8, 3) if name != "A" else (-4, 8)
    ax.annotate(f"{name} ({int(end[0])}, {int(end[1])})", end,
                xytext=offset, textcoords="offset points", color=color, weight="bold")
ax.set(xlim=(-0.7, 12.3), ylim=(-0.8, 22.6),
       xlabel="Count of 'data'", ylabel="Count of 'learning'",
       title="A and B point in the same direction")
ax.grid(alpha=0.17)
plt.show()

### Why the two measures tell different stories

From the results:

- **Euclidean distance:** A is much closer to **C**, because C's absolute counts are similar to A's.
- **Cosine similarity:** A is most similar to **B** because B is exactly 10 times A and points in the same direction.
- Neither is automatically “correct”: one compares **absolute differences**, while the other emphasizes **direction**.

Cosine similarity is itself a **similarity score**, not a mathematical distance metric. There are ways to define related angular distances, but we do not need them here.

**A practical caution:** Cosine similarity is often helpful for document vectors, but it is not automatically suitable for penguin measurements. The meaning of the **origin**, signs, and the chosen representation matters. For instance, standardizing measurements moves the origin to a training mean; an angle in that space may answer a different question from the one we intended.

**Next question:** Physical measurements and word counts are numerical. Our penguins also have an **island** name. Does Euclidean distance make sense for a categorical label?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">07 · CATEGORICAL INFORMATION AND FEATURE CHOICE</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Being numerically encodable does not guarantee meaningful distance.</div>
</div>

### Is Biscoe twice as far as Dream?

The island values **Biscoe, Dream, Torgersen** are *nominal categories*. If we encode them as `0`, `1`, and `2` and calculate numerical differences, we would invent an ordering and a spacing that the data do not contain.

A simpler categorical comparison is **matching**:

$$
d_{\mathrm{match}}(a,b)=
\begin{cases}
0 & \text{if the categories are the same},\\
1 & \text{if they are different}.
\end{cases}
$$

For several categorical features, **Hamming distance** counts positions at which categories differ (sometimes reported as a proportion). This is a different idea from a straight-line measurement.

In L08 we used **one-hot encoding** so island membership could be represented numerically. Let's see what such a representation actually means for Euclidean distance.

In [ ]:
# Use only island categories found in the illustrative training observations.
training_islands = sorted(study.loc[training_rows, "island"].dropna().unique())
assert set(study["island"].dropna().unique()) <= set(training_islands), (
    "An unseen category would require a documented handling policy"
)
island_cat = pd.Categorical(study["island"], categories=training_islands)
island_onehot = pd.get_dummies(
    pd.Series(island_cat, index=study.index, name="island"),
    prefix="island", dtype=int
)

print("Training-category vocabulary:", training_islands)
display(pd.concat([study.loc[[0, 6, 145], ["island"]],
                   island_onehot.loc[[0, 6, 145]]], axis=1))

print("Matching distance, island of row 0 versus row 6:",
      int(study.loc[0, "island"] != study.loc[6, "island"]))
print("Matching distance, island of row 0 versus row 145:",
      int(study.loc[0, "island"] != study.loc[145, "island"]))
print("One-hot Euclidean distance, rows 0 and 145:",
      round(np.linalg.norm(island_onehot.loc[0].to_numpy() -
                           island_onehot.loc[145].to_numpy()), 3))

### A numeric encoding does not settle how important island should be

With **complete one-hot encoding**, two distinct islands differ in **two indicator columns**: one changes from 1 to 0 and another from 0 to 1.

Therefore, their one-hot Euclidean distance is $\sqrt{1^2+(-1)^2}=\sqrt{2}$. Identical islands have distance **0**.

These are properties of the **encoding**, not measured geographic distances.

What if we combine standardized physical measurements with these indicators?

- We can concatenate the vectors and calculate Euclidean distance.
- But this gives island indicators a **particular numerical weight** relative to the standardized measurements.
- That weight is a **choice**, not something proven by the data. The meaning of the category and the intended question still matter.

In [ ]:
# The four standardized physical measurements come from Section 05.
physical_z = numeric_z[MEASUREMENTS].copy()
physical_and_island = pd.concat([physical_z, island_onehot], axis=1)

# Species and split roles are NOT included as distance features.
assert "species" not in physical_and_island.columns
assert "split_role" not in physical_and_island.columns
assert physical_and_island.isna().sum().sum() == 0

physical_distances = euclidean_to_reference(physical_z)
mixed_distances = euclidean_to_reference(physical_and_island)
physical_top5 = physical_distances.drop(index=0).sort_values(kind="stable").head(5)
mixed_top5 = mixed_distances.drop(index=0).sort_values(kind="stable").head(5)

rankings = pd.DataFrame({
    "Four standardized physical features": physical_top5.index.to_list(),
    "Four physical + island indicators": mixed_top5.index.to_list(),
}, index=pd.Index(range(1, 6), name="Rank"))
display(rankings)

print("Closest with physical measurements only:", physical_top5.index[0])
print("Closest after also including island:", mixed_top5.index[0])
print("Reference island:", study.loc[0, "island"])
print("New closest observation's island:", study.loc[mixed_top5.index[0], "island"])

In [ ]:
# Why rows 145 and 6 change order under the two representations.
focus_rows = [145, 6]
fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.6), constrained_layout=True)
for ax, distances, title in [
    (axes[0], physical_distances, "Four standardized physical features"),
    (axes[1], mixed_distances, "Also include one-hot island indicators")
]:
    values = distances.loc[focus_rows]
    bars = ax.bar(["Row 145", "Row 6"], values,
                  color=["#557AB5", TEAL], width=0.55)
    ax.bar_label(bars, fmt="%.3f", padding=3)
    ax.set(title=title, ylabel="Numerical Euclidean distance", ylim=(0, values.max() * 1.24))
    ax.grid(axis="y", alpha=0.16)
plt.show()

**Interpretation:** For reference row 0, the closest observation under four standardized physical measurements is **row 145**. After we add island indicators, the closest becomes **row 6**, which shares the reference island.

We have not discovered that row 6 is biologically more similar. We have **changed what the comparison considers important**.

The same warning applies to derived or repeated numerical features. If we include **bill length, bill depth, and their ratio**, we allow bill-related information to contribute several times. More features do not necessarily mean more independent evidence.

In [ ]:
# Optional illustration within the discussion: adding a derived feature changes distances.
ratio = study["bill_length_mm"] / study["bill_depth_mm"]
ratio_train_mean = ratio.loc[training_rows].mean()
ratio_train_sd = ratio.loc[training_rows].std(ddof=1)
assert ratio_train_sd > 0

physical_with_ratio = physical_z.copy()
physical_with_ratio["bill_shape_ratio_z"] = (ratio - ratio_train_mean) / ratio_train_sd
ratio_top5 = (euclidean_to_reference(physical_with_ratio)
              .drop(index=0).sort_values(kind="stable").head(5))

print("Four physical features, closest five rows:", physical_top5.index.tolist())
print("Four features + derived bill ratio, closest five:", ratio_top5.index.tolist())
print("These are alternative representations, not evidence of better prediction.")

### What about sets and other kinds of data?

Some observations describe **sets of present attributes** rather than ordinary measured quantities. For example, documents might be represented by the set of words they contain.

- **Jaccard similarity** compares the size of the shared part of two sets with the size of their union: $|A\cap B|/|A\cup B|$ (for a nonempty union).
- It can be useful for **set or binary-presence data**, particularly when shared absences should not count as evidence of similarity.

**What else can affect proximity?** Outliers, correlated/redundant features, and many dimensions can change the usefulness of distance. Standardization alone does not resolve these problems.

**Connection to later lectures:** Proximity will eventually support nearest-neighbor classification (L15) and clustering (L19)

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">08 · SYNTHESIS AND LOOKING AHEAD</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">The formula is only one part of defining similarity.</div>
</div>

### What changed our answer today?

We discovered that **similarity depends not only on the distance measure, but also on the features we choose to compare**.

- Euclidean and Manhattan distance can rank the same observations differently.
- Units and scaling can change numerical proximity, even when the observations are unchanged.
- Cosine similarity emphasizes direction rather than magnitude.
- Categorical indicators and derived features change which differences receive attention.
- Closeness in a selected feature space does **not** prove two observations share the same label.

**Our next question:**

> **If the features we choose can change how we understand our data, how can we decide which features are truly useful, which are redundant, and whether we can represent the same information with fewer features?**

This leads to **L10 — Feature Selection, PCA, and Reliable Preprocessing Pipelines**, where we will explore how to select, simplify, and organize feature representations for Machine Learning.

### Sources, acknowledgments, and further reading

- **Palmer Penguins:** Horst, A. M., Hill, A. P., & Gorman, K. B. (2020), *palmerpenguins: Palmer Archipelago (Antarctica) penguin data* ([project](https://allisonhorst.github.io/palmerpenguins/); [Zenodo DOI](https://doi.org/10.5281/zenodo.3960218)). Field observations: Gorman, K. B., Williams, T. D., & Fraser, W. R. (2014), *PLoS ONE*. The numerical comparisons in this notebook use the course's locally provided CSV.
- **Teaching inspirations:** Selected concepts were adapted from the teaching material of **Prof. Antonio Liotta**.
- **NumPy:** [`np.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html) and [`np.dot`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html).
- **SciPy:** [distance measures](https://docs.scipy.org/doc/scipy/reference/spatial.distance.html), including Euclidean, cityblock/Manhattan, Minkowski, Hamming, and Jaccard (reference only; SciPy is not required to run this notebook).
- **Scikit-learn:** [pairwise metrics and cosine similarity](https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity) (reference only; no model or Scikit-learn code is required).

**Offline execution:** The code needs only NumPy, Pandas, Matplotlib, IPython/Jupyter, and the existing `data/penguins.csv`. It does not require internet access.